# Data-analytiikan perusteet II: ryhmittely, pivot-taulukot, yhdistäminen ja datan siivous

Tässä notebookissa jatkamme ensimmäisen notebookin kaupunkipyöräaineistolla. Harjoittelemme datan siivousta, tietojen tiivistämistä ryhmittäin ja kahden taulukon yhdistämistä. 

## Jupyter Notebook lyhyesti

Jupyter Notebook on työkirja, jossa teksti, Python-koodi ja koodin tuottamat tulokset ovat samassa tiedostossa. Sisältö on jaettu **soluihin**, tekstisolut sisältävät ohjeita ja koodisoluissa suoritetaan ohjelmakoodia.

- Valitse koodisolu napsauttamalla sitä. **Shift + Enter** suorittaa solun ja siirtää valinnan seuraavaan soluun. Tulokset, kuten taulukot ja kuvaajat, näkyvät solun alapuolella.
- Etene koodisoluissa **ylhäältä alas**, sillä myöhemmät solut käyttävät aiemmissa soluissa luotuja muuttujia. Kun muutat koodia, suorita solu ja siitä riippuvat solut uudelleen.
- Voit muokata tekstisolua kaksoisnapsauttamalla sitä. **Shift + Enter** näyttää tekstin jälleen muotoiltuna.
- Koodia suorittava **ydin (kernel)** pitää muuttujat muistissa. Jos käynnistät ytimen uudelleen, suorita myös aiemmat koodisolut uudelleen.
- Tallenna muokkauksesi File -> Save a copy in drive

## Oppimistavoitteet

Kun olet käynyt notebookin läpi, osaat:

- tunnistaa ja poistaa duplikaatteja
- muuntaa tekstimuotoisia arvoja numeroiksi sekä päivämääriksi ja ajoiksi
- käsitellä puuttuvia ja virheellisiä havaintoja
- tiivistää tietoja `groupby()`- ja `agg()`-metodeilla
- rakentaa ja tulkita pivot-taulukoita
- yhdistää kaksi DataFramea `merge()`-metodilla
- erottaa `left`- ja `inner`-liitoksen käyttötarkoitukset
- koota siivouksen vaiheet uudelleen käytettäväksi funktioksi

## Avoin data, muokkaukset ja lisenssi

Notebook perustuu kahteen HSL:n avoimeen aineistoon:

1. **Helsingin ja Espoon kaupunkipyörillä ajetut matkat**  
   https://hri.fi/data/fi/dataset/helsingin-ja-espoon-kaupunkipyorilla-ajatut-matkat
2. **HSL:n kaupunkipyöräasemat**  
   https://hri.fi/data/fi/dataset/hsl-n-kaupunkipyoraasemat

Molemmat aineistot on julkaistu **Creative Commons Attribution 4.0 (CC BY 4.0)** -lisenssillä. Matkadatan ylläpitäjä on HSL ja alkuperäinen tekijä / datan omistaja City Bike Finland. Asemadatan ylläpitäjä on HSL.

Tässä notebookissa käytetään opetukseen rajattuja otoksia molemmista aineistoista. Dataan on synteettisesti lisätty yleisimpiä analytiikassa vastaan tulevia ongelmia, sekä niistä on jätetty sisältöä pois. Tiedosto `hsl_matkat_2021_07_siivous.csv` sisältää puuttuvia ja virheellisiä arvoja sekä toistuvia rivejä.

## 1. Aineistojen lukeminen

Käytämme ensimmäisestä notebookista tuttua aineistoa ja lisäksi pientä HSL:n otosta kaupunkipyöräasemista. Asemataulu sisältää aseman tunnisteen, nimen, kapasiteetin ja koordinaatit.

Luemme aineistot jälleen CSV-tiedostoista `pd.read_csv()`-funktiolla. Tallennamme matkat muuttujaan `raaka`, jotta voimme myöhemmin verrata siivottua aineistoa lähtötilanteeseen.

In [2]:
import pandas as pd

# Luetaan raakadata ja nimetään sarakkeet kuten ensimmäisessä notebookissa.
raaka = pd.read_csv('https://raw.githubusercontent.com/ratuom/ohjelmoinnin-perusteet-data-analytiikka/main/hsl_matkat_2021_07_siivous.csv').rename(columns={
    'Departure': 'lahto_aika',
    'Return': 'paluu_aika',
    'Departure station id': 'lahtoasema_id',
    'Departure station name': 'lahtoasema',
    'Return station id': 'paluuasema_id',
    'Return station name': 'paluuasema',
    'Covered distance (m)': 'matka_m',
    'Duration (sec.)': 'kesto_s'
})

# Luetaan asemien tiedot erilliseen DataFrame-taulukkoon.
asemat = pd.read_csv('https://raw.githubusercontent.com/ratuom/ohjelmoinnin-perusteet-data-analytiikka/main/hsl_asemat_otos.csv')

print('Raakadatan rivejä:', len(raaka))
print('Asemia opetuksen asemataulussa:', len(asemat))

Raakadatan rivejä: 41
Asemia opetuksen asemataulussa: 24


In [3]:
# Näytetään matkataulukon ensimmäiset viisi riviä.
raaka.head()

,lahto_aika,paluu_aika,lahtoasema_id,lahtoasema,paluuasema_id,paluuasema,matka_m,kesto_s
0,2021-07-31T23:59:59,2021-08-01T00:09:15,113,Pasilan asema,78,Messeniuksenkatu,1602,553
1,2021-07-31T23:59:55,2021-08-01T00:08:45,135,Velodrominrinne,115,Venttiilikuja,1307,532
2,2021-07-31T23:59:55,2021-08-01T00:03:24,258,Abraham Wetterin tie,260,Herttoniemi (M),820,205
3,2021-07-31T23:59:47,2021-08-01T00:05:52,122,NaN,16,Liisanpuistikko,1298,369
4,2021-07-31T23:59:33,2021-08-01T00:14:49,126,Kalasatama (M),255,Laivalahden puistotie,3875,912


In [4]:
# Näytetään asemataulukon ensimmäiset viisi riviä.
asemat.head()

,station_id,nimi,kapasiteetti,pituusaste,leveysaste
0,11,Unioninkatu,22,24.951023,60.167457
1,16,Liisanpuistikko,16,24.961375,60.174182
2,113,Pasilan asema,40,24.932799,60.198211
3,115,Venttiilikuja,16,24.941523,60.194192
4,116,Linnanmäki,32,24.940159,60.191141


Raakadatassa on 41 riviä ja asemataulukossa 24 asemaa. Matkataulukon yksi rivi kuvaa matkaa, kun taas asemataulukon yksi rivi kuvaa asemaa. Raakadatan rivimäärä ei vielä kerro erillisten matkojen määrää, sillä sama rivi voi toistua aineistossa.

Ensimmäisissä riveissä näkyy jo puuttuva lähtöaseman nimi. Tarkistamme seuraavaksi koko aineiston, jotta myös muualla olevat puutteet tulevat esiin.

# Osa I - Datan siivous

Dataa siivotessamme emme välttämättä halua poistaa kaikkea epätavallista aineistosta. Ensin pitää määritellä, **mikä analyysin kannalta on virhe**. Tässä harjoituksessa käytämme selkeitä sääntöjä, asiat eivät ole aina näin yksinkertaisia ja vaativat useasti syvällisempää tutkimista. 

## 2. Tarkista ennen kuin muutat

Niin kuin ensimmäisessä notebookissakin, aloitamme ymmärtämällä mitä data oikeastaan on syönyt. Tiedämme dokumentaation perusteella mitä aineiston pitäisi sisältää. Näimme kuitenkin jo yhdellä rivillä puuttuvan lähtöaseman, joten tiedämme että ainakin yksi olettamus datan eheydestä ei pidä paikkansa. 

Tarkastetaan seuraavaksi:

- rivien määrän
- tietotyypit
- duplikaatit
- puuttuvat arvot

In [ ]:
print('Rivejä:', len(raaka))
print('\nTietotyypit:')
print(raaka.dtypes)
print('\nTäysin identtisiä duplikaatteja:', raaka.duplicated().sum())
print('\nPuuttuvat arvot:')
print(raaka.isna().sum())

Tulosteessa näkyy 41 riviä, kaksi duplikaattia ja yksi puuttuva lähtöaseman nimi. Aineistossa piilottelee kuitenkin vielä lisää virheitä, jotka eivät ole välittömästi selviä. Pandasin object ei kerro meille juurikaan sen sisällöstä, sillä se voi olla lähes mitä vaan. Ensimmäisessä notebookissa jätimme asian käsittelemättä, mutta hyvien tapojen mukaista on muuttaa muuttujat mahdollisimman tarkaksi tietomuodoksi ongelmien välttämiseksi. Tässä tarvitsemme tyyppimuunnoksia.

## 3. Tyyppimuunnokset ja virheelliset arvot

Matkojen pituuksilla ja kestoilla pitää voida laskea, ja aikoja pitää voida verrata keskenään. Muunnamme siksi nämä sarakkeet sopiviin tietotyyppeihin.

`pd.to_numeric(..., errors='coerce')` yrittää muuntaa arvon numeroksi. Jos muunnos ei onnistu, tulokseksi tulee puuttuva arvo `NaN`.

Vastaavasti `pd.to_datetime(..., errors='coerce')` muuntaa kelvolliset aikaleimat päivämäärä- ja aikatyypiksi ja virheelliset arvot puuttuviksi `NaT`-arvoiksi. Muunnos ei korjaa virheellistä sisältöä, vaan tekee sen näkyväksi seuraavaa vaihetta varten.

In [ ]:
siisti = raaka.copy()

# Muunnetaan matkan pituus ja kesto numeroiksi.
siisti['matka_m'] = pd.to_numeric(siisti['matka_m'], errors='coerce')
siisti['kesto_s'] = pd.to_numeric(siisti['kesto_s'], errors='coerce')

# Muunnetaan lähtö- ja paluuajat päivämäärä- ja aikatyypiksi.
siisti['lahto_aika'] = pd.to_datetime(siisti['lahto_aika'], errors='coerce')
siisti['paluu_aika'] = pd.to_datetime(siisti['paluu_aika'], errors='coerce')

print(siisti.dtypes)
print('\nPuuttuvat arvot muunnosten jälkeen:')
print(siisti.isna().sum())

Puuttuvia arvoja on nyt yksi lähtöajassa, lähtöaseman nimessä, matkan pituudessa ja kestossa. Muunnokset paljastivat siis kolme uutta puutetta. Tämä osoittaa, miksi pelkkä `isna()` ei riitä raakadatassa. 

## 4. Duplikaattien poistaminen

Saman matkan ylimääräinen kopio kasvattaisi sekä matkojen määrää että kokonaispituutta. Aineistosta löytyi kaksi täysin identtistä duplikaattiriviä. Poistamme ylimääräiset kopiot `drop_duplicates()`-metodilla.

In [ ]:
print('Ennen:', len(siisti))
print('Duplikaatteja:', siisti.duplicated().sum())

# Säilytetään kustakin täysin samanlaisesta rivistä ensimmäinen.
siisti = siisti.drop_duplicates()
print('Jälkeen:', len(siisti))

Rivimäärä pienenee 41:stä 39:ään. Kummastakin toistuvasta rivistä poistui ylimääräinen kopio, ja yksi jäi aineistoon.

## 5. Puuttuvien ja mahdottomien arvojen käsittely

Tässä analyysissä matka tarvitsee lähtö- ja paluuajan, asemien nimet, pituuden sekä keston. Jos jokin näistä puuttuu, rivi jätetään analyysistä pois.

Lisäksi asetamme seuraavat yksinkertaiset laatuehdot:

- matkan pituus > 0 m
- kesto > 0 s
- paluuaika ei voi olla ennen lähtöaikaa

Nämä ovat tämän harjoituksen ehtoja, eivätkä ne sellaisenaan sovellu läheskään kaikkiin tilanteisiin. Esimerkiksi matkan pituus 0 m voisi olla kiinnostava havainto sen sijaan, että se poistettaisiin automaattisesti. Se voisi viitata esimerkiksi käyttäjän kohtaamaan ongelmaan, vialliseen laitteistoon tai systemaattiseen virheeseen datan keräämisessä.

Jos esimerkiksi useiden matkojen pituudeksi olisi tallentunut 0 m, vaikka lähtö- ja pääteasema olisivat eri asemia ja muut tiedot vaikuttaisivat olevan kunnossa, olisi syytä epäillä jonkin olevan pielessä.

Tällaisen virheen merkitys voisi kasvaa huomattavasti, jos matkan pituutta käytettäisiin suoraan esimerkiksi laskutuksessa. Jos hinta laskettaisiin kaavalla (avausmaksu + matkan pituus × hinta) olisi näissä virhetilanteissa kyseessä aika hyvä diili asiakkaalle. 

In [ ]:
kriittiset = [
    'lahto_aika', 'paluu_aika', 'lahtoasema', 'paluuasema', 'matka_m', 'kesto_s'
]

# Poistetaan rivit, joilta puuttuu jokin analyysissä tarvittava arvo.
siisti = siisti.dropna(subset=kriittiset)

# & tarkoittaa, että kaikkien kolmen ehdon pitää toteutua samalla rivillä.
siisti = siisti[
    (siisti['matka_m'] > 0) &
    (siisti['kesto_s'] > 0) &
    (siisti['paluu_aika'] >= siisti['lahto_aika'])
].copy()

print('Siivouksen jälkeen rivejä:', len(siisti))

### Tarkista siivouksen tulos

Jäljelle jää 34 matkaa: neljä riviä poistui puuttuvien tietojen vuoksi ja yksi nollan metrin pituuden vuoksi. Ensimmäisessä notebookissa säilytimme nollan metrin matkan tarkastelua varten. Tässä se poistetaan valitun analyysisäännön perusteella.

Tarkistamme vielä puuttuvat arvot, duplikaatit ja pienimmät arvot, jotta näemme, toteutuivatko asetetut laatuehdot.

In [ ]:
print('Puuttuvia kriittisissä sarakkeissa:', siisti[kriittiset].isna().sum().sum())
print('Täysiä duplikaatteja:', siisti.duplicated().sum())
print('Pienin matka (m):', int(siisti['matka_m'].min()))
print('Pienin kesto (s):', int(siisti['kesto_s'].min()))

Puuttuvien kriittisten arvojen ja duplikaattien määrät ovat nyt nollia. Lyhin matka on 14 metriä ja lyhin kesto 61 sekuntia, joten myös pienimmät arvot ovat positiivisia. Hyvin lyhyt matka voisi silti vaatia lisätarkastelua analyysin tarkoituksesta riippuen.

# Osa II - Ryhmittely ja aggregointi

Aggregointi tarkoittaa havaintojen tiivistämistä esimerkiksi summiksi, lukumääriksi tai keskiarvoiksi. Yksittäiset rivit kertovat yksittäisistä matkoista. Usein tarvitaan kuitenkin vastauksia ryhmätasolla: **kuinka monta matkaa asemalta lähti, mikä oli keskimääräinen pituus tai kuinka paljon matkaa kertyi yhteensä?**

## 6. `groupby()` - jaa, laske, yhdistä

`groupby()` voidaan ajatella kolmena vaiheena:

1. data jaetaan ryhmiin
2. jokaiselle ryhmälle tehdään sama laskenta
3. tulokset yhdistetään taulukoksi

Selvitetään ensin, **miltä lähtöasemilta siivotun otoksen matkat lähtivät useimmin**. `size()` laskee ryhmän rivit, ja `sort_values(ascending=False)` järjestää tulokset suurimmasta pienimpään.

In [ ]:
# Ryhmitellään lähtöaseman mukaan ja lasketaan matkojen määrä.
matkoja_asemittain = siisti.groupby('lahtoasema').size().sort_values(ascending=False)
matkoja_asemittain.head(10)

Abraham Wetterin tie on edelleen yleisin lähtöasema, mutta matkoja on nyt neljä ensimmäisen notebookin viiden sijaan. Ero johtuu siitä, että nollan metrin matka poistettiin siivouksessa.

### Yhden numeerisen sarakkeen aggregointi

Matkojen määrä ei vielä kerro, kuinka paljon asemalta lähteneillä matkoilla ajettiin. Lasketaan siksi kunkin lähtöaseman matkojen kokonaispituus. Ryhmittelyn jälkeen valitsemme `matka_m`-sarakkeen ja laskemme sen arvot yhteen `sum()`-metodilla.

In [ ]:
siisti.groupby('lahtoasema')['matka_m'].sum().sort_values(ascending=False).head(10)

Tuloksen luvut ovat nyt metrejä. Ensimmäisenä on Abraham Wetterin tie, jonka matkojen yhteispituus on 14 830 metriä. `head(10)` näyttää vain kymmenen suurinta kokonaispituutta.

### Useita tunnuslukuja samalla kertaa

`agg()` laskee useita tunnuslukuja samalla ryhmittelyllä. Näin voimme verrata asemien matkamääriä, kokonaispituuksia ja tyypillisiä matkoja yhdessä taulukossa.

Esimerkiksi `keskipituus_m=('matka_m', 'mean')` luo sarakkeen nimeltä `keskipituus_m` laskemalla `matka_m`-sarakkeen keskiarvon kussakin ryhmässä.

In [ ]:
# Kootaan jokaisesta lähtöasemasta yksi rivi ja neljä tunnuslukua.
asema_yhteenveto = (
    siisti.groupby('lahtoasema')
    .agg(
        matkoja=('matka_m', 'size'),
        matkaa_yhteensa_m=('matka_m', 'sum'),
        keskipituus_m=('matka_m', 'mean'),
        mediaanikesto_s=('kesto_s', 'median')
    )
    .sort_values('matkoja', ascending=False)
)

asema_yhteenveto.head(10)

Yksi rivi kokoaa nyt yhden lähtöaseman matkat. Esimerkiksi Abraham Wetterin tien neljän matkan kokonaispituus on 14 830 metriä ja keskipituus 3707,5 metriä. `mediaanikesto_s` kertoo keston mediaanin sekunteina.

# Osa III - Pivot-taulukot

Pivot-taulukko on ohjelmallinen vastine laskentataulukon pivotille. Se tiivistää dataa kahden tai useamman luokittelevan muuttujan mukaan.

## 7. Valmistellaan kaksi luokkaa pivot-taulukkoa varten

Haluamme verrata eripituisten matkojen määriä sen mukaan, liittyykö matka metroasemaan. Tarvitsemme tähän kaksi luokittelevaa saraketta.

Ensimmäisessä notebookissa luokittelimme pituudet omalla funktiolla ja `.apply()`-metodilla. Nyt käytämme `pd.cut()`-funktiota, joka jakaa arvot annettuihin luokkaväleihin. `right=False` sisällyttää alarajan ja jättää ylärajan pois, joten luokat ovat samat kuin aiemmin:

- alle 1500 m → `lyhyt`
- vähintään 1500 m mutta alle 3000 m → `keskipitkä`
- vähintään 3000 m → `pitkä`

Metroaseman tunnistamme nimessä olevasta `(M)`-merkinnästä kuten ensimmäisessä notebookissa. `.map()` muuntaa haun totuusarvot luokiksi `kyllä` ja `ei`.

In [ ]:
# Luokitellaan matkat pituuden perusteella. float('inf') antaa viimeiselle luokalle rajattoman ylärajan.
siisti['pituusluokka'] = pd.cut(
    siisti['matka_m'],
    bins=[0, 1500, 3000, float('inf')],
    labels=['lyhyt', 'keskipitkä', 'pitkä'],
    right=False
)

# | yhdistettynä regexeihin tarkoittaa, että lähtö- tai paluuaseman nimestä löytyy (M)-merkintä.
siisti['metro_matka'] = (
    siisti['lahtoasema'].str.contains(r'\(M\)', regex=True) |
    siisti['paluuasema'].str.contains(r'\(M\)', regex=True)
).map({True: 'kyllä', False: 'ei'})

siisti[['matka_m', 'pituusluokka', 'metro_matka']].head()

Taulukosta voit verrata matkan pituutta sille annettuun luokkaan. Esimerkiksi 820 metrin matka on lyhyt ja sen metro-luokka on `kyllä`, koska matka päättyy metroasemalle. Luokka ei siis kerro vain lähtöasemasta.

### Pivot 1: määrä kahden kategorian mukaan

Pivot-taulukosta näemme yhdellä silmäyksellä, miten matkat jakautuvat pituusluokan ja metromatkojen mukaan.

- `index` valitsee taulukon rivien luokat ja `columns` sarakkeiden luokat.
- `values` valitsee sarakkeen, jonka arvoista laskenta tehdään.
- `aggfunc='count'` laskee tämän sarakkeen arvot ilman puuttuvia arvoja. Siivotussa `matka_m`-sarakkeessa jokainen matka tulee mukaan.
- `fill_value=0` täyttää tyhjät yhdistelmät nollilla.
- `observed=False` ottaa mukaan myös pituusluokat, joissa ei olisi havaintoja.

In [ ]:
# Lasketaan matkojen määrä kussakin pituusluokan ja metro-luokan yhdistelmässä.
pivot_maarat = pd.pivot_table(
    siisti,
    index='pituusluokka',
    columns='metro_matka',
    values='matka_m',
    aggfunc='count',
    fill_value=0,
    observed=False
)

pivot_maarat

**Tulkinta:** jokainen solu kertoo yhden luokkayhdistelmän matkamäärän. Esimerkiksi lyhyitä metroasemaan liittyviä matkoja on kaksi. Keskipitkien matkojen `kyllä`-sarakkeen nolla tarkoittaa, ettei tähän yhdistelmään kuulu yhtään matkaa. Kaikkien solujen summa on siivotun aineiston 34 matkaa.

### Pivot 2: keskimääräinen matkan pituus

Matkamäärien lisäksi voimme verrata ryhmien keskipituuksia. Säilytämme taulukon rakenteen ja vaihdamme laskennaksi `aggfunc='mean'`. `.round(1)` pyöristää tulokset yhden desimaalin tarkkuuteen.

Keskiarvotaulukon puuttuva arvo jätetään näkyviin: jos ryhmässä ei ole matkoja, sen keskipituus ei ole nolla metriä.

In [ ]:
pd.pivot_table(
    siisti,
    index='pituusluokka',
    columns='metro_matka',
    values='matka_m',
    aggfunc='mean',
    observed=False
).round(1)

**Tulkinta:** taulukon arvot ovat nyt keskipituuksia metreinä. Lyhyiden metroasemaan liittyvien matkojen keskipituus on 417 metriä. Keskipitkien matkojen `kyllä`-sarakkeessa näkyy `NaN`, koska tähän ryhmään ei kuulu matkoja, joista keskiarvo voitaisiin laskea.

# Osa IV - Kahden taulukon yhdistäminen

Matkataulussa on aseman tunniste ja nimi, mutta ei esimerkiksi aseman kapasiteettia tai koordinaatteja. Ne ovat erillisessä asemataulussa. Tämä on tietokannoissa ja aineistoissa erittäin tavallista.

## 8. Avaimet ennen liitosta

Liitos tarvitsee yhteisen **avaimen**, jonka avulla saman aseman tiedot tunnistetaan eri taulukoissa. Matkataulun `lahtoasema_id` vastaa asemataulun `station_id`-saraketta.

Tarkistamme ennen liitosta, että kukin asematunniste esiintyy asemataulussa vain kerran. Muuten yksi matka voisi saada useita osumia ja monistua liitoksen tuloksessa.

In [ ]:
print('Asemataulun rivejä:', len(asemat))
print('Yksilöllisiä station_id-arvoja:', asemat['station_id'].nunique())
print('Duplikaatti-ID:t:', asemat['station_id'].duplicated().sum())

Asemataulukossa on 24 riviä ja 24 erilaista tunnistetta, eikä tunnisteissa ole duplikaatteja. Kukin lähtöaseman tunniste voi siis löytää tästä taulukosta enintään yhden vastineen.

## 9. Left join - vasen liitos

`how='left'` säilyttää **kaikki vasemman taulukon rivit**. Käytämme sitä, kun haluamme lisätä matkoihin asematiedot ja säilyttää myös matkat, joiden lähtöasemalle ei löydy vastinetta. Tässä asemataulussa on vain osa asemista, joten osalle matkoista lisättäviin sarakkeisiin tulee puuttuvia arvoja.

`left_on` ja `right_on` kertovat, mistä sarakkeista yhteistä avainta etsitään. `validate='many_to_one'` varmistaa, että useat matkat saavat liittyä samaan asemaan, mutta asemataulun tunnisteella on vain yksi rivi.

In [ ]:
# Lisätään jokaiseen matkaan sen lähtöaseman tiedot, jos ne löytyvät asemataulusta.
yhdistetty = siisti.merge(
    asemat,
    left_on='lahtoasema_id',
    right_on='station_id',
    how='left',
    validate='many_to_one',
    indicator=True
)

print(yhdistetty['_merge'].value_counts())
yhdistetty[['lahtoasema_id', 'lahtoasema', 'kapasiteetti', '_merge']].head(12)

`indicator=True` lisää `_merge`-sarakkeen. Arvo `both` tarkoittaa, että avain löytyi molemmista taulukoista. `left_only` tarkoittaa, että matka säilyi, mutta tämän otoksen asemataulussa ei ollut kyseistä asemaa.

**Tulkinta:** asematiedot löytyvät 23 matkalle, ja 11 matkaa jää ilman osumaa. Kaikki 34 matkaa ovat silti mukana. `right_only`-rivien määrä on nolla, sillä vasen liitos ei lisää mukaan pelkkiä asemataulukon rivejä.

## 10. Inner join - sisäliitos

`inner` säilyttää vain rivit, joille löytyy vastine molemmista taulukoista. Tämä on hyödyllistä, jos analyysi vaatii ehdottomasti asematiedot, mutta samalla havaintoja voi pudota pois.

In [ ]:
# Säilytetään vain matkat, joiden lähtöasema löytyy asemataulusta.
sisaliitos = siisti.merge(
    asemat,
    left_on='lahtoasema_id',
    right_on='station_id',
    how='inner',
    validate='many_to_one'
)

print('Left join -rivejä:', len(yhdistetty))
print('Inner join -rivejä:', len(sisaliitos))

**Tulkinta:** vasen liitos säilyttää 34 matkaa ja sisäliitos 23 matkaa. Ero on 11 matkaa, joiden lähtöasemaa ei ole tässä asemataulukossa. Liitostavan valinta vaikuttaa siis siihen, mitkä matkat päätyvät jatkoanalyysiin.

### Mitä yhdistetystä datasta voidaan selvittää?

Nyt voimme verrata matkoja esimerkiksi lähtöasemien kapasiteetin eli pyöräpaikkojen määrän mukaan. Katsotaan ensin, mitä tietoja liitos lisäsi. `.notna()` valitsee rivit, joilla kapasiteetti ei ole puuttuva arvo.

In [ ]:
# Näytetään matkan pituus sekä lähtöaseman kapasiteetti ja koordinaatit.
yhdistetty.loc[
    yhdistetty['kapasiteetti'].notna(),
    ['lahtoasema', 'matka_m', 'kapasiteetti', 'pituusaste', 'leveysaste']
].head(10)

Samalle lähtöasemalle kuuluvilla matkoilla kapasiteetti toistuu samana. Se on aseman ominaisuus, eikä sitä pidä laskea yhteen matkariveiltä aseman kokonaiskapasiteetiksi.

Kuten edellisestä notebookista muistetaan, `describe()` kokoaa sarakkeen tunnusluvut yhteen taulukkoon. Kun sitä kutsutaan yhdelle sarakkeelle, tulos on tavallinen pandasin Series. Sitä voi siis indeksoida samalla tavalla kuin muitakin Series-olioita, esimerkiksi indeksistä `['max']`.

Jos `describe()` kutsutaan usealle sarakkeelle, tulos on DataFrame: rivit ovat tunnuslukuja ja sarakkeet alkuperäisen aineiston sarakkeita. Pelkkä `['max']` etsisi silloin saraketta nimeltä `max` ja antaisi virheen. Tunnusluvun rivi valitaan `.loc[]`-valinnalla, ja siihen voi lisätä myös sarakkeen nimen.

In [ ]:
# Tulostetaan yhdistetyn datan matkan pituuden tunnusluvut.
kuvaus = yhdistetty['matka_m'].describe()
print(kuvaus,'\n')

# Haetaan yksittäiset tunnusluvut nimellä.
print('Pisin matka (m):', int(kuvaus['max']))
print('Matkan pituuden mediaani (m):', int(kuvaus['50%']),'\n')

# Lasketaan tunnusluvut kahdelle sarakkeelle. Tulos on DataFrame.
kuvaus_df = yhdistetty[['matka_m', 'kesto_s']].describe()
print(kuvaus_df,'\n')

# Valitaan max-rivi: tulos on Series, jossa on kummankin sarakkeen suurin arvo.
print(kuvaus_df.loc['max'])

# Valitaan rivi ja sarake: tulos on yksittäinen luku.
print('Pisin matka (m):', int(kuvaus_df.loc['max', 'matka_m']))

# Osa V - Toistettava siivousketju

Kun samoja aineistoja käsitellään myöhemmin uudelleen, siivouksen vaiheita ei kannata tehdä käsin yksi kerrallaan. Kokoamme siksi aiemmat vaiheet funktioksi, joka tekee samat muunnokset ja rajaukset jokaisella kutsulla.

Funktio käyttää luvussa 5 määriteltyä `kriittiset`-listaa. Se palauttaa siivotut perussarakkeet; pituus- ja metro-luokittelut teimme erikseen myöhemmässä vaiheessa.

In [ ]:
def siivoa_matkat(data):
    # Käsitellään kopiota, jotta funktion saama alkuperäinen taulukko säilyy.
    tulos = data.copy()

    # Muunnetaan numerot ja ajat samoilla säännöillä kuin aiemmin.
    tulos['matka_m'] = pd.to_numeric(tulos['matka_m'], errors='coerce')
    tulos['kesto_s'] = pd.to_numeric(tulos['kesto_s'], errors='coerce')
    tulos['lahto_aika'] = pd.to_datetime(tulos['lahto_aika'], errors='coerce')
    tulos['paluu_aika'] = pd.to_datetime(tulos['paluu_aika'], errors='coerce')

    # Poistetaan ylimääräiset kopiot ja kriittisiä tietoja vailla olevat rivit.
    tulos = tulos.drop_duplicates()
    tulos = tulos.dropna(subset=kriittiset)

    # Säilytetään vain matkat, jotka täyttävät kaikki laatuehdot.
    tulos = tulos[
        (tulos['matka_m'] > 0) &
        (tulos['kesto_s'] > 0) &
        (tulos['paluu_aika'] >= tulos['lahto_aika'])
    ].copy()
    return tulos

# Kutsutaan funktiota raakadatalla ja verrataan rivimäärää aiempaan tulokseen.
uusi_siisti = siivoa_matkat(raaka)
print('Funktion tulos:', len(uusi_siisti), 'riviä')
print('Sama rivimäärä kuin aiemmin:', len(uusi_siisti) == len(siisti))

Funktio palauttaa 34 riviä, kuten vaiheittainen siivouskin. Tulosteen `True` kertoo rivimäärien olevan samat. Se ei kuitenkaan vielä yksin osoita, että myös taulukoiden kaikki arvot olisivat samoja.

# Harjoitukset

Vastaa kysymyksiin tässä notebookissa esitettyjen menetelmien avulla. Voit käyttää alla olevaa tyhjää solua omiin laskuihisi. Vastaukset ovat joko **kokonaislukuja** tai **merkkijonoja**.

**Tehtävät 1-3** harjoittavat tulosten lukemista: vastaukset löytyvät suoraan aiempien koodiesimerkkien tuloksista.

**Tehtävät 4-8** vaativat soveltamista: yhdistä aiemmin läpikäytyjä komentoja ja kirjoita oma ratkaisusi. Vihjeitä löydät tarvittaessa yhteenvedon jälkeen, yritä kuitenkin ratkaista tehtävät ensin ilman.

**Käytä tehtävissä 3-8 siivottua aineistoa ja siitä muodostettuja taulukoita.**

1. Kuinka monta täysin identtistä duplikaattiriviä `raaka`-DataFramessa on ennen siivousta?  
2. Kuinka monta riviä jää lopulliseen `siisti`-DataFrameen?  
3. Mikä lähtöasema esiintyy siivotussa aineistossa useimmin? Kirjoita aseman nimi täsmälleen datan mukaisesti.  
4. Kuinka monta metriä ajettiin yhteensä asemalta **Abraham Wetterin tie** lähteneillä vähintään 3000 metriä pitkillä matkoilla? Anna vastaus kokonaislukuna.  
5. Kuinka monta alle 3000 metriä pitkää matkaa on sellaisia, joiden kummankaan aseman nimessä ei esiinny merkintää `(M)`? Käytä `pivot_maarat`-taulukkoa.  
6. Kuinka monta erilaista lähtöaseman tunnistetta on matkoissa, joille vasen liitos ei löytänyt asematietoja?  
7. Minkä lähtöaseman kapasiteetti on suurin niistä yhdistetyistä riveistä, joille kapasiteetti löytyy? Anna nimi `lahtoasema`-sarakkeen mukaisesti.  
8. Kuinka monta erilaista lähtöaseman tunnistetta on matkoissa, joille asematiedot löytyivät?

In [ ]:
# Kirjoita oma koodisi tähän.

## Yhteenveto

Tässä notebookissa siivosimme aineistoa, tiivistimme matkoja lähtöasemittain, rakensimme pivot-taulukoita ja yhdistimme matkat erilliseen asematauluun. Näillä yksinkertaisilla työkaluilla saamme jo yllättävän paljon tietoa irti datasta. 

## Vihjeet harjoituksiin

4. **Rajattujen matkojen yhteispituus:** yhdistä lähtöaseman nimeä ja matkan pituutta koskevat ehdot `&`-operaattorilla. Muista sulut kummankin ehdon ympärille. Suodata `siisti`-taulukko ja laske `matka_m`-sarakkeen arvot yhteen `sum()`-metodilla.
5. **Kahden pituusluokan yhteismäärä:** alle 3000 metrin matkat ovat luokissa `lyhyt` ja `keskipitkä`. Valitse nämä rivit ja sarake `ei` pivot-taulukosta `.loc[]`-valinnalla ja laske määrät yhteen.
6. **Ilman asematietoja jääneet lähtöasemat:** suodata `yhdistetty`-taulukosta rivit, joilla `_merge` on `left_only`. Laske `lahtoasema_id`-sarakkeen erilaisten arvojen määrä `nunique()`-metodilla.
7. **Suurimman kapasiteetin lähtöasema:** selvitä `kapasiteetti`-sarakkeen suurin arvo `describe()`-tuloksen `max`-riviltä kuten ensimmäisen notebookin pisimmän matkan tehtävässä. Suodata tämän kapasiteetin rivit ja tarkastele `lahtoasema`-saraketta.
8. **Asematiedot saaneet lähtöasemat:** valitse `yhdistetty`-taulukosta rivit, joilla `_merge` on `both`, ja käytä `lahtoasema_id`-sarakkeen `nunique()`-metodia. Voit myös hyödyntää `sisaliitos`-taulukkoa, jossa jokaiselle matkalle löytyi asematiedot.